# Score Payment Transaction

This notebook provides a web-style interactive form for the **Score Payment** scenario only. It reuses the same scoring engine as the Streamlit app: `score_encrypted_transaction`.

## Browser Access

To open this interactive notebook as a browser web page, start it with Voilà from the project root:

```bash
python3 -m voila src/score_payment_tx_ui.ipynb --port 8866 --Voila.ip=127.0.0.1 --no-browser
```

Then open this URL in Chrome:

[http://127.0.0.1:8866/](http://127.0.0.1:8866/)

In [1]:
from pathlib import Path
import base64
import json
import sys

project_root = Path.cwd()
if (project_root / "src").exists():
    sys.path.insert(0, str(project_root / "src"))
else:
    sys.path.insert(0, str(project_root))

from algorithms.crypto_algorithms import ALGORITHM_PROFILES, score_encrypted_transaction

try:
    import ipywidgets as widgets
    from IPython.display import HTML, display, clear_output
except ImportError as exc:
    raise ImportError("Install ipywidgets to use this interactive notebook: pip install ipywidgets") from exc

## Interactive Score Payment Form

Choose a payment scenario, edit the fields, then click **Score Payment**.

In [2]:
SAMPLE_TRANSACTIONS = {
    "Legacy high-value SWIFT": {
        "txn_id": "TXN-20240301-00192",
        "algorithm": "RSA-2048",
        "channel": "SWIFT_MT103",
        "amount_sgd": 2_850_000.0,
        "payload": base64.b64encode(b"legacy-rsa512-ciphertext-demo-payment-record").decode("utf-8"),
    },
    "PQC protected SWIFT": {
        "txn_id": "TXN-20240301-00988",
        "algorithm": "ML-KEM-768",
        "channel": "SWIFT_MT103",
        "amount_sgd": 2_850_000.0,
        "payload": (
            "f4a7c9e2b8d60144aa31f0dbe9368a22f10593d984bc6a73"
            "71ec0fb5346329af68a80bd8e8f7b908c9dbe6719041cfbc"
        ),
    },
    "Signed ACH authorization": {
        "txn_id": "TXN-20240301-00193",
        "algorithm": "ML-DSA-65",
        "channel": "ACH_CREDIT",
        "amount_sgd": 450_000.0,
        "payload": base64.b64encode(b"ml-dsa-signature-demo-for-ach-payment-authorization").decode("utf-8"),
    },
}

risk_colors = {
    "LOW": "#157347",
    "MEDIUM": "#997404",
    "HIGH": "#b54708",
    "CRITICAL": "#b42318",
}

display(HTML("""
<style>
.score-card {
    border: 1px solid #e4e7ec;
    border-radius: 8px;
    padding: 16px;
    background: #f8fafc;
    margin-top: 12px;
}
.score-grid {
    display: grid;
    grid-template-columns: repeat(4, minmax(120px, 1fr));
    gap: 12px;
    margin-bottom: 14px;
}
.metric {
    border: 1px solid #e4e7ec;
    border-radius: 8px;
    background: #ffffff;
    padding: 10px 12px;
}
.metric-label { color: #475467; font-size: 12px; }
.metric-value { color: #101828; font-size: 22px; font-weight: 700; }
.recommendation {
    border-left: 4px solid #2563eb;
    background: #eff6ff;
    padding: 10px 12px;
    margin-top: 12px;
}
</style>
"""))

sample_picker = widgets.Dropdown(
    options=list(SAMPLE_TRANSACTIONS.keys()),
    value="PQC protected SWIFT",
    description="Scenario",
    layout=widgets.Layout(width="520px"),
)
txn_id_input = widgets.Text(description="Txn ID", layout=widgets.Layout(width="520px"))
algorithm_input = widgets.Dropdown(
    options=list(ALGORITHM_PROFILES.keys()),
    description="Algorithm",
    layout=widgets.Layout(width="520px"),
)
channel_input = widgets.Dropdown(
    options=["SWIFT_MT103", "ACH_CREDIT", "CARD", "WIRE", "PAYMENT"],
    description="Channel",
    layout=widgets.Layout(width="520px"),
)
amount_input = widgets.FloatText(description="SGD", layout=widgets.Layout(width="260px"))
payload_input = widgets.Textarea(
    description="Payload",
    layout=widgets.Layout(width="100%", height="150px"),
)
score_button = widgets.Button(description="Score Payment", button_style="primary", icon="shield")
output = widgets.Output()

def load_sample(sample_name):
    sample = SAMPLE_TRANSACTIONS[sample_name]
    txn_id_input.value = sample["txn_id"]
    algorithm_input.value = sample["algorithm"]
    channel_input.value = sample["channel"]
    amount_input.value = float(sample["amount_sgd"])
    payload_input.value = sample["payload"]

def render_result(result):
    risk_color = risk_colors.get(result["risk_rating"], "#344054")
    findings = "".join(f"<li>{finding}</li>" for finding in result["findings"])
    html = f"""
    <div class="score-card">
      <div class="score-grid">
        <div class="metric"><div class="metric-label">Security Score</div><div class="metric-value">{result['score']}/100</div></div>
        <div class="metric"><div class="metric-label">Risk Rating</div><div class="metric-value" style="color:{risk_color};">{result['risk_rating']}</div></div>
        <div class="metric"><div class="metric-label">Quantum Safe</div><div class="metric-value">{'Yes' if result['quantum_safe'] else 'No'}</div></div>
        <div class="metric"><div class="metric-label">Valid Encoding</div><div class="metric-value">{'Yes' if result['valid_encoding'] else 'No'}</div></div>
      </div>
      <p><strong>Algorithm:</strong> {result['algorithm']} &nbsp; <strong>Entropy:</strong> {result['entropy']}</p>
      <p><strong>Findings</strong></p>
      <ul>{findings}</ul>
      <div class="recommendation"><strong>Recommendation:</strong> {result['recommendation']}</div>
    </div>
    """
    display(HTML(html))
    print("Raw JSON result:")
    print(json.dumps(result, indent=2))

def score_payment(_=None):
    result = score_encrypted_transaction(
        encrypted_payload=payload_input.value,
        algorithm_name=algorithm_input.value,
        amount_sgd=amount_input.value,
        channel=channel_input.value,
    )
    with output:
        clear_output(wait=True)
        print(f"Transaction: {txn_id_input.value} | Channel: {channel_input.value} | Amount: SGD {amount_input.value:,.2f}")
        render_result(result)

def on_sample_change(change):
    if change["name"] == "value":
        load_sample(change["new"])
        score_payment()

sample_picker.observe(on_sample_change, names="value")
score_button.on_click(score_payment)
load_sample(sample_picker.value)

form = widgets.VBox([
    sample_picker,
    widgets.HBox([txn_id_input, amount_input]),
    widgets.HBox([algorithm_input, channel_input]),
    payload_input,
    score_button,
    output,
])

display(form)
score_payment()